In [2]:
using LinearAlgebra, BenchmarkTools, PolynomialRoots, StaticArrays, DataStructures

# Fully-split velocity Lagrangian PDMP

## Splitting equations and general rate solution
Consider, for simplicity, a general split PDMP with each transitions $\alpha \to \beta$, i.e. only the split dynamic changes (Other events are entirely possible to consider in the same framework, but clutter up notation, so let us leave them out for now).  Let $\lambda(\alpha \to \beta)$ be the associated rates. To preserve a distribution $\mu$ the split flow is taken to satisfy

$
\mu^{-1}\text{div}_\alpha(\mu \Phi^\alpha) = \sum_\beta  \lambda(\alpha \to \beta) - \lambda(\alpha \to \beta)
$

and denoting the divergence (choice of sign is non-obvious for A) on the LHS by $A_\alpha$, and $\lambda_{\alpha\beta} =\lambda(\alpha \to \beta) $ we get

$
A_\alpha = \sum_\beta \lambda_{\alpha\beta}-\lambda_{\beta\alpha}
$

Of course, if the original $\Phi$ satisfies $\text{div}(\mu \Phi)=0$ then so does the sum of split flows, as the sum of divergences $A_\alpha$ vanish by linearity of the divergence operator.

In trying to define the rates we have to solve a constraint on an anti-symmetric matrix. Clearly, adding any symmetric part to $\lambda_{\alpha\beta}$ is inconsequential for the constraint above. Effectively, under such a transformation, the altered flow ${\alpha \to \beta}$ is offset by an equivalent flow $\beta \to \alpha$, and in simulation we will get more or fewer events but the overall divergence is not altered. Since events are typically costly we generally will want $\lambda_{{\alpha\beta}}$ to have as small a symmetric part as possible. Alas, $\lambda_{{\alpha\beta}} \geq 0 $ so the symmetric part cannot be zero identically.

Rather than solving the above positivity and anti-symmetry constraints on $\lambda_{{\alpha\beta}}$ we can assume 

$
\lambda_{{\alpha\beta}} = [\rho_{\alpha\beta}]^+
$

where $\rho_{\alpha\beta}$ is some anti-symmetric matrix. Then 

$
\lambda_{\alpha\beta} - \lambda_{\beta\alpha}= [\rho_{\alpha\beta}]^+ - [\rho_{\beta\alpha}]^+ = [\rho_{\alpha\beta}]^+-[-\rho_{\alpha\beta}]^+ = \rho_{\alpha \beta}
$

and thus we have to solve

$
A_\alpha = \sum_\beta \rho_{\alpha\beta}
$

where the $A_\alpha$ are determined by $\text{div}_\alpha(\mu \Phi_\alpha)$. A simple solution is given by

$
\rho_{\alpha \beta} = (A^\alpha-A^\beta)/n
$

where $n$ is the number of split states. This solution gives us a direct interpretation of the rates: it is possible to transition into a state $\beta$ _precisely_ when the divergence of flow associated to $\beta$ exceeds the divergence of the flow in the current state $\alpha$. The greater the difference in divergence, the more likely we are to transition into the given state.


Note that the $A_\alpha$ are arbitrary (but must satisfy $\sum_\alpha A_\alpha=0$), and so this is a valid solution and rates for any split. If for some reason we want to have a substantially increased rate we may adjust the rate matrix $\lambda_{\alpha\beta} \to \lambda_{\alpha\beta} + R_{\alpha\beta}$ where $R_{\alpha\beta}$ is any positive definite symmetric matrix. 

As we shall soon see there are several other solutions of interest.

## Preferential splitting rates
Suppose we are interested in a particular state, say, $\alpha = 0$, for which we want to have as small rates $\lambda_{0\beta}$ as possible. In other words, we would like 'stay' in the state $\alpha = 0$ for longer (or, at the very least, have fewer events $0 \to \beta$). Then, as we shall see, the above rates are not ideal. The rate $\lambda_{0 \to \text{any}} = \sum_\beta \lambda_{0\beta}$
becomes

$\lambda_{0 \to \text{any}} = \frac{1}{n+1}\sum_\beta [A_0-A_\beta]^+\geq \frac{1}{n+1}[\sum_\beta A_0-A_\beta]^+ = \frac{1}{n+1}[\sum_\beta A_0]^+ = [A_0]^+$

with equality if and only if $A_0 - A_\beta \geq 0$ for every $\beta$.

Had we only split into $0$ and taken the other splits $\beta= 1,2,\ldots, n $ as a single state $I$ the above 'recipe' would instead declare

$\lambda_{0 \to I} = \frac{1}{2}[A_0 - A_I]^+ =[A_0]^+$

since $A_0 + A_I = 0$. Clearly the former choice means that we transition into $I$ with a frequency that is *at least* as big as that of the latter choice, but generally larger.

Thus, if possible, we would like to use another rate for transitions $0 \to \beta$. Mercifully there are some obvious choices. Considering the equation

$A_0 = \sum_\beta \lambda_{0\beta}-\lambda_{\beta 0 }$

we can, as above, pick an anti-symmetric $\rho$-matrix and $\lambda_{0 \beta} = [\rho_{0\beta}]^+$ and $\lambda_{0 \beta} = [-\rho_{0\beta}]^+$  which leads to

$A_0 = \sum_\beta \rho_{0\beta}$

which we can solve by letting $\rho_{0\beta} = \frac{1}{n} A_0 = -\rho_{\beta 0}$ for $\beta \neq 0$. How should we interpret this choice for the rates? Now we only shift from $\alpha =0$ to _some_ (note that we effectively pick which uniformly at random) $\beta$ if the average divergence in $-A_\beta = A_0$ exceeds $0$, rather than if some individual divergence does so.

This has some consequence for the divergence equations for other splits. Let lower-case latin letters $a \neq 0$ index the $n$ splits other than $\alpha = 0$. We can in fact apply the same recipe as above again. To see this we have

$A_a = \sum_\alpha \lambda_{a\alpha}-\lambda_{\alpha a} =  (\lambda_{a0}-\lambda_{0a}) + \sum_b\lambda_{ab}-\lambda_{b a}  = \rho_{a0} + \sum_b\lambda_{ab}-\lambda_{b a} 
=-\frac{A_0}{n} + \sum_b\rho_{ab}$

Thus, picking $\rho_{ab} = \frac{1}{n}(A_a - A_b)$ we get

$ \sum_b\rho_{ab} = A_a - \frac{1}{n}\sum_b A_b = A_a  - \frac{1}{n}(-A_0 + \sum_{\beta} A_\beta ) = A_a + \frac{1}{n}A_0$

since $\sum_b A_b = -A_0 + \sum_\beta A_\beta = -A_0$ due to the vanishing overall divergence. Hence the $A_0$ terms in $a$-divergence equation cancel and the solution

$\rho_{ab} = \frac{1}{n}(A_a - A_b)$ 

is satisfactory.

### The Lagrangian split-velocity case
We now turn our attention to a splitting of the Lagrangian dynamics, so that we have one state corresponding to evolution of position, and one state for each evolution of a velocity component $v^i$. Thus, in sampling in $\mathbb{R}^n$ we shall have $n+1$ split states. In our sampling we shall treat position updates preferentially (
Shifting between velocities is *hopefully* inexpensive, but shifting between position and velocity updates is quite costly as it incurs a computation of third order derivatives.).

We shall, in what comes, adopt a somewhat specialized notation. We let the position flow correspond to the split state variable $\alpha = n+1$, so that the $i$:th velocity component can be chosen to be represented by $\alpha = i$, and so that matrix-enumerations match this (Matrices and arrays in Julia are, of course - as with any other sane language - indexed starting from 1.).



## Ricatti equation and the velocity flow
The fully split velocity satisfies the Ricatti equation

$du/dt = a u^2 + b u + c$

for some $a,b,c$. This admits the solution

$u(t) = (\kappa \tan(\kappa(t+t_0))-(b/2))/a$

where $\kappa = \sqrt{4ac-b^2}/2$

Notably, if $\kappa$ is imaginary $\kappa = i k$ for some $k \in \mathbb{R}$ then

$\kappa \tan(\kappa s) = i k \tan( iks ) = -k\text{tanh}(ks)$

In [60]:
riccati(t, a, b, c; t0 = 0.0) = a*c-(b/2)^2 < 0 ? ricatti_κ(t, a, b, sqrt(abs(a*c-(b/2)^2)), t0 = t0, imag = true) : ricatti_κ(t, a, b, sqrt((a*c-(b/2)^2)), t0 = t0, imag = false)
ricatti_κ(t, a, b, κ; t0 = 0.0, imag::Bool = false) = imag ? (κ * tan(κ*(t+t0))-(b/2))/a : (-κ * tanh(κ*(t+t0))-(b/2))/a

@benchmark ricatti_κ($0.2, $1.0, $2.0, $3.0)

BenchmarkTools.Trial: 10000 samples with 996 evaluations per sample.
 Range (min … max):  19.980 ns … 178.815 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     20.683 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   22.870 ns ±   7.038 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▆  ▁   ▃▂▁        ▁▁                                       ▁
  ███▆▆█▇▅████▄▄▅▅▅▅▆███▅▆▆▆▆▆▆▇▅▄▄▅▃▄▅▅▇████▇▇▆▅▅▄▂▃▅▄▂▃▅▄▅▆▅ █
  20 ns         Histogram: log(frequency) by time      48.2 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

## Rate integrals and rates
### Exact methods
The rates are given by some expression

$\lambda^{IJ} = [\rho^{IJ}]^+ = [A^I - A^J]^+/n$

and the total rate in state $I$ is simply the sum $\lambda^I = \sum_J\lambda^{IJ}$. The $A^I$ are cubic in $u(t)$ so each $\lambda^{IJ}$ is some cubic polynomial in $u(t)$. We can explicitly integrate each $\lambda^{IJ}$ *if* we know that it is positive. Thus, for a given $I$, we solve $\rho^{IJ}(u) = 0$ for all $J$ and order the individual solutions $u_1, u_2, \ldots$ such that $u_i < u_{i+1}$ if $du/dt > 0$, and $u_i> u_{i+1}$ else (Note that $du/dt$ is actually identically positive or negative for all $t$ we will consider, since $u(t)$ 'blows up' in finite time (this is a weird argument - it is true because we know that $u(t)$ has to blow up in finite time, and has to be a solution of the Ricatti equation)). This partitions the velocity space $U$ into sets over which the signs of all $\rho^{IJ}$ are constant and over these the sum $\sum_J \lambda^{IJ}$ can thus be computed.

In [4]:
pol = [9,3,-7,1]
@benchmark roots($pol)

BenchmarkTools.Trial: 10000 samples with 196 evaluations per sample.
 Range (min … max):  475.000 ns … 77.896 μs  ┊ GC (min … max): 0.00% … 98.83%
 Time  (median):     517.857 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   642.777 ns ±  1.075 μs  ┊ GC (mean ± σ):  7.77% ±  5.74%

  ▅██▅▅▄▅▅▃▃▃▃▃▂▂▁   ▁    ▁▂▂▂▁                                ▂
  █████████████████████████████████▇▇▇▇▇▆▆▆▇▅▆▅▇▆▅▆▆▅▆▁▅▅▄▆▅▄▅ █
  475 ns        Histogram: log(frequency) by time      1.51 μs <

 Memory estimate: 496 bytes, allocs estimate: 8.

In [5]:
"""
    real_roots(b, c, d; verbose=false)::Tuple

Finds the real roots of the cubic polynomial x^3 + b x^2 +c x + d.
"""
function real_roots(b, c, d; verbose=false)::Tuple
    Δ = b^2 - 3*c
    μ = (2*(b^3)) - (9*b*c) + (27*d)

    if iszero(Δ) && iszero(μ) #Should check vs threshold.
        return (-b/3,) 
    end

    #Optimize: Add special statement for when Δ = 0 (within threshold.)
    L = (μ^2)-(4*Δ^3)
    if L < 0 
        verbose ? println("L < 0") : nothing
        z = (μ + sqrt(-L)*im) #2z = ...,  but we only use the angle for z
        r2 = cbrt((μ^2 - L)/4)
        if r2 ≈ Δ #This must be handled more carefully!
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing

            (s, c) = sincos(angle(z)/3) #can be optimized/altered to use the 'tan-formula' for the 3xReal root cubic
            k = 2*sqrt(r2)
            return (b .+ (k.* (c, (-c + (sqrt(3)*s))/2, (-c - (sqrt(3)*s))/2))) ./(-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            if μ ≤ 0
                verbose ? println("μ ≤ 0") : nothing
                return (b + sqrt(r2)*(1+(Δ/(r2))),) ./(-3)
            else
                verbose ? println("μ > 0") : nothing
                return (b - sqrt(r2)*(1+(Δ/(r2))),)./(-3)
            end
        end
    elseif L ≥ 0
        verbose ? println("L ≥ 0, L: $L") : nothing
        root = sqrt(L)
        if μ > 0
            z = (μ + root)/2
        else
            z = (μ - root)/2
        end
            
        C = cbrt(z)
        r2 = C^2
        if r2 ≈ Δ #This must be handled more carefully! We should check roots (amounts to a single computation) 
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing
            return (b+2*C, b-C) ./ (-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            return (b + C*(1+(Δ/r2)),) ./(-3)
        end
    end
end

real_roots(a,b,c,d) = real_roots(b/a, c/a, d/a)

real_roots (generic function with 2 methods)

Thus we have established a pretty decent root-finder. Let's look at its performance:

In [6]:
bm_roots_1R = @benchmark real_roots($2.0, $(-3.0), $9.0) #One real root

BenchmarkTools.Trial: 10000 samples with 987 evaluations per sample.
 Range (min … max):  55.117 ns … 283.384 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     58.359 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   61.245 ns ±  10.769 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▂█▁█▂▁▁    ▁▂▃▃▂▁         ▁       ▁▂▂▁                      ▂
  █████████▇████████████▇▇▇▇███▇▇▇▇█▇█████▇▇▇▆▅▇▇▇▆▆▆▆▅▁▃▅▅▅▆▆ █
  55.1 ns       Histogram: log(frequency) by time      99.9 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [7]:
bm_roots_3R = @benchmark real_roots($(-7.0), $(3.0), $9.0) #Three real roots

BenchmarkTools.Trial: 10000 samples with 962 evaluations per sample.
 Range (min … max):  83.784 ns … 985.655 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     88.462 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   97.753 ns ±  28.824 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅██▃▄▅▆▂▁▂▃▂   ▁          ▁▁▂▁                               ▂
  █████████████████████▇▇▇▇▇██████▇▆▆▇▇▇▆▇▇▆▅▅▆▇▅▅▅▄▄▅▅▄▄▄▄▄▃▃ █
  83.8 ns       Histogram: log(frequency) by time       202 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

This is something like $\sim 10-15$ times faster than the one defined above. As a rough measure of performance then we expect something like $\sim 80 N ns$ to compute the partition of the time interval for a velocity update. For e.g. the 20-dimensional Twin Peaks scenario $N = 21$ so we get something akin to $\sim 1.6 \mu s$ of work. Naturally there may be other f Of course, if we wanted to we could parallellize this particular task, but the overhead for such an endeavour would probably be prohibitive. 

#### Partitioning the time intervals

It shall become convenient to define $A^I(u) = \sum_j A^I_j u^j $ and $\rho^{IJ} = \sum_j\rho^{IJ}_ju^j$.

In [18]:
real_roots(X::SVector{4, Float64}) = real_roots(X[1], X[2], X[3], X[4])
real_roots(X::AbstractArray) = real_roots(X[1], X[2], X[3], X[4])

real_roots (generic function with 4 methods)

In [13]:
A1=@SVector rand(4)
@benchmark real_roots($A1) 

BenchmarkTools.Trial: 10000 samples with 983 evaluations per sample.
 Range (min … max):  60.224 ns … 656.663 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     62.055 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   67.115 ns ±  14.505 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▂▇▄▃▃▁  ▁▁▂▂▁▁                    ▁▁▁                      ▁
  ████████▇▇███████▇▇▇▇▇████▇▇▇▆▆▇▆▇▆████▆▆▆▆▇▆▇▇▅▆▃▅▃▄▅▃▅▆▅▄▄ █
  60.2 ns       Histogram: log(frequency) by time       114 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [ ]:
function partition_by_roots!(T::BinaryMinHeap{Float64}, I::Integer, ρ::Array{Float64, 3}, a, b, κ; start_time = 0.0) #could be $MVector
    empty!(T)
    #push!(T, start_time)
    if isreal(κ)
        t(u) = atan((a*u + (b/2))/κ) - start_time 
    else
        k = imag(κ)
        t(u) = atanh((a*u + (b/2))/k) - start_time
    else 
    ρ_I = @view ρ[I,:,:]
    for J in axes(ρ_I, 1)
        roots = real_roots(@view(ρ_I[J,:]))
        for root in roots
            #The root corresponds to a velocity for which the rate becomes positive.
            #We transform it into a time.
            root_time = t(root)  
            if root_time > start_time
                push!(T, root_time)
            end
        end
    end
    return T
end            

partition_by_roots! (generic function with 2 methods)

In [53]:
D = 21
ρ = rand(21, 21, 4);
T = BinaryMinHeap{Float64}()
a = rand()
b = rand()
κ = rand()
T = partition_by_roots!(T, 1, ρ, a, b, κ)

BinaryMinHeap{Float64}(Base.Order.ForwardOrdering(), [0.0646566825098035, 0.06674727496756434, 0.09645307765931627, 0.10817817357348904, 0.19500516798777628, 0.13622429943322648, 0.14703759495982607, 0.240693679199773, 0.14399285128732822, 0.20679076916441042  …  0.27979536955963735, 0.25440760598753515, 0.16962439602321436, 0.266659800863187, 0.2579088433483795, 0.3097600673495254, 0.2982114940450635, 0.24658678857778876, 0.2709406127948546, 0.3227799015199161])

In [54]:
@benchmark partition_by_roots!($T, $1, $ρ, $a, $b, $κ)

BenchmarkTools.Trial: 10000 samples with 9 evaluations per sample.
 Range (min … max):  2.133 μs …  51.478 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.222 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.427 μs ± 874.790 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▆█▂                 ▃▄▄▃▂▁▁▂▂                             ▂
  ██████▆▇█▇▇▇▅▆▆▇▅▇▆█▇███████████▇▆▆▇▇▆▅▅▆▆▅▅▄▄▅▇▆▆▆▅▄▄▁▅▄▅▆ █
  2.13 μs      Histogram: log(frequency) by time      4.27 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

We get roughly to $2\mu s$ of computation time, indicating that the transformation of each root takes a slight amount of time.

#### The rate integrals 
The rates need to be integrated, that is we integrate

$\lambda = A u^3 + B u^2 + C u + D = \bar{A} \cdot \bar{U}$

over time (on the intervals discussed above). Each integral

$M_n = \int u(t)^n dt$

can be analytically computed. We expand in the $\tan(\kappa (t+t_0))$ after a transformation $s = \kappa(t+t_0)$ so that

$dt = ds/\kappa$

and (by abuse of notation)

$u(s) = (\kappa \tan(s) -(b/2))/a$

whence

$M_n(s) = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} (-b/2)^{n-j} \int (\kappa \tan(s))^{j}ds = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} \kappa^{j}(-b/2)^{n-j} L_{j} = \frac{1}{a^n\kappa}\bar{M_n} \cdot \bar{L}$

with each $L_j$ corresponding to a $\tan(x)^j$ integral. It can be shown that

1) $L_0 = s$
2) $L_1 = -\log(\cos(s))$
3) $L_2 = (\tan(s)-s)$
3) $L_3 = (\frac{1}{2\cos^2(s)} + \log(\cos(s)))$

Obviously there are some issues with the values - they can be divergent, complex and otherwise problematic. However, we shall always have $s$ in specific domains, and the integrals shall only be integrated over domains for which $\lambda \geq 0$.

Of course we shall typically evaluate this at many distinct "times" $s$, which alters the vector $L$. Thus it makes sense to use a matrix $M$ with the distinct $M_i$ as rows so that $\bar{U}(s) = M \bar{L}(s)$

In [ ]:
function L_tuple(s::Float64)
    c = cos(s)
    lc = log(c)
    return (s, -lc, tan(s) -s , lc + (1. /(2*(c^2))))
end

L_tuple (generic function with 1 method)

In [84]:
function M_matrix!(M::MMatrix{4,4, Float64, 16}, a, b, κ)
    for i in 1:4
        for j in 1:i
            M[i, j] = binomial(i, j) * ((-b / 2.)^j) * (κ^(i-j))
        end
        factor = (κ * (a^i))
        @views M[i,:] ./= factor
    end
    return M 
end

M_matrix! (generic function with 4 methods)

In [85]:
M = @MMatrix zeros(4,4)
M_matrix!(M, rand(), randn(), rand())

4×4 MMatrix{4, 4, Float64, 16} with indices SOneTo(4)×SOneTo(4):
  -2.70791   0.0        0.0       0.0
  -5.33126   4.71954    0.0       0.0
  -7.87201  13.9375    -8.22554   0.0
 -10.3321   27.4398   -32.3884   14.336

In [86]:
@benchmark M_matrix!($M, $a, $b, $κ)

BenchmarkTools.Trial: 10000 samples with 421 evaluations per sample.
 Range (min … max):  236.817 ns …  1.196 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     251.069 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   280.751 ns ± 74.558 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▇▃▃▄▂▁▂▃▃▄▁▄▁▁▁▁                                           ▂
  █████████████████████▇▇▇▇▇▇▆▆▇▆▆▆▅▆▅▇█▇▇███████▇▆▆▅▆▆▆▅▆▄▅▅▄ █
  237 ns        Histogram: log(frequency) by time       576 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [ ]:
function compute_rate_integrals(ρ_IJs::Vector{SVector{4, Float64}}, M::MMatrix, initial_values::Vector{Float64}, s_final::Float64)
    